# Predict this week

Set `SEASON` and `WEEK`, then run all cells. The table is the predicted winner, win probability, and the biggest reasons (style matchup, weather, injuries, form).

First run downloads nflverse data and can take several minutes. Later runs use the local cache.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from nfl_predictor.config import current_nfl_season
from nfl_predictor.models.predict import default_week, predict_week
from nfl_predictor.models.train import ensure_model
from nfl_predictor.pipeline import build_feature_table

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

SEASON = current_nfl_season()
WEEK = None  # None = next unplayed week
SEASON, WEEK

In [ ]:
features = build_feature_table()
if WEEK is None:
    WEEK = default_week(features, SEASON)
print(f"Predicting {SEASON} week {WEEK}")
artifact = ensure_model(features)
preds = predict_week(features, season=SEASON, week=WEEK, artifact=artifact)
preds

In [ ]:
if preds.empty:
    available = (
        features.loc[features["season"] == SEASON, "week"]
        .dropna()
        .astype(int)
        .sort_values()
        .unique()
        .tolist()
    )
    print(f"No games found for {SEASON} week {WEEK}. Available weeks: {available}")
else:
    view = preds.copy()
    view["matchup"] = view["away_team"] + " @ " + view["home_team"]
    view["weather"] = view.apply(
        lambda r: "indoor" if r.get("is_indoor") else f"{r.get('temp')} F, wind {r.get('wind')}",
        axis=1,
    )
    show = view[
        [
            "gameday",
            "matchup",
            "away_style",
            "home_style",
            "away_injury_penalty",
            "home_injury_penalty",
            "weather",
            "predicted_winner",
            "predicted_win_prob",
            "reasons",
        ]
    ]
    display(show)

Probabilities are P(that team wins the game). They will not sum to anything across the slate; each game is independent. If a game already has a final score, you can compare `predicted_winner` to the actual result in the first table.